# Sequential dilution, step 1 of 3 — build subgroups

Splits the spectra at each concentration into `M` subgroups of `N_seq` spectra
and writes both the raw subgroups and their averages. Sampling is without
replacement while `M * N_seq` fits inside the available spectra and with
replacement beyond that.

**Input** — one ML-format CSV covering all viruses and concentrations.

**Output** — one folder per virus, holding each subgroup's CSV, its averaged
spectrum, the index list that produced it, and a validation report.

**Next** — `02_extract_combinations.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from datetime import datetime

# Section 1: Setup & Configuration

In [ ]:
# ===========================================================================
# Section 1: Setup & Configuration
# ===========================================================================

# ==================== FIXED PARAMETERS ====================
K = 10  # Number of spectra per subgroup (controls SNR - NEVER change across train/test)
N = 2   # Number of sequential dilutions (concentration pairs)

# ==================== DATASET-SPECIFIC PARAMETERS ====================
dataset_type = 'test'  

if dataset_type == 'train':
    M = 10  # Number of subgroups per concentration → 400 combinations (20×20)
elif dataset_type == 'val' or dataset_type == 'test' or dataset_type == 'unknown_test':
    M = 7   # Number of subgroups per concentration → 49 combinations (7×7)
    # M = 4   # Number of subgroups per concentration for preliminary result → 16 combinations (4x4)
else:
    raise ValueError("dataset_type must be 'train' or 'test'")

# ==================== SAMPLING STRATEGY ====================
ALLOW_REPLACEMENT = True
SAMPLING_STRATEGY = 'force_include_first'  # Strategy A: ensures all spectra appear at least once
RANDOM_SEED = 42  # For reproducibility (set to None for truly random)

# ==================== VIRUS LIST ====================
# List all viruses to process
virus_list = ['B1', 'B1351', 'B16172', 'BA5', 'EG51', 'JN1', 'SARSCoV2', 'XBB15', 'ProbeDNA']

print(f"Viruses to process: {virus_list}")

# ==================== FILE PATHS ====================
DATA_FOLDER = "/home/zhao/Jiaheng Cui/DNA RNA hybridization/data"
INPUT_CSV = os.path.join(DATA_FOLDER, f"03042026-{dataset_type}-seed42-despike_airPLS_mean_01scale_mean.csv")  # Main data file

# Output folder with timestamp and parameters
timestamp = '03042026'
OUTPUT_FOLDER = f"/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results/{timestamp}-{dataset_type}-K{K}_M{M}-subgroup_creation"

# Create main output directory
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ==================== TRACKING STRUCTURES ====================
# Track processing results for each virus
virus_processing_log = {
    'successful': [],
    'failed': [],
    'warnings': []
}

# Track statistics per virus
virus_stats = {}  # Will store: {virus_name: {concentrations, subgroups, combinations, etc.}}

# ==================== CONFIGURATION SUMMARY ====================
print("="*70)
print("MODIFIED CODE 1: MULTI-VIRUS SUBGROUP BUILDER")
print("="*70)
print(f"\n📊 CONFIGURATION:")
print(f"   Dataset type: {dataset_type.upper()}")
print(f"   K (spectra per subgroup): {K}")
print(f"   M (subgroups per concentration): {M}")
print(f"   N (sequential dilutions): {N}")
print(f"   Expected combinations per pair: {M}×{M} = {M*M}")
print(f"\n🔄 SAMPLING STRATEGY:")
print(f"   Method: {SAMPLING_STRATEGY}")
print(f"   Replacement: {ALLOW_REPLACEMENT}")
print(f"   Random seed: {RANDOM_SEED}")
print(f"\n🦠 VIRUSES TO PROCESS:")
for i, virus in enumerate(virus_list, 1):
    print(f"   {i}. {virus}")
print(f"   Total: {len(virus_list)} viruses")
print(f"\n📁 FILE PATHS:")
print(f"   Input CSV: {INPUT_CSV}")
print(f"   Output folder: {OUTPUT_FOLDER}")

# Verify input file exists
if os.path.exists(INPUT_CSV):
    print(f"   ✅ Input file found")
else:
    print(f"   ❌ Input file NOT found: {INPUT_CSV}")
    print(f"   Please check the path and try again!")
    raise FileNotFoundError(f"Input file not found: {INPUT_CSV}")

# Verify output folder created
if os.path.exists(OUTPUT_FOLDER):
    print(f"   ✅ Output folder created")
else:
    print(f"   ❌ Failed to create output folder")
    raise OSError(f"Could not create output folder: {OUTPUT_FOLDER}")

print(f"\n✅ Section 1 complete. Ready to process {len(virus_list)} viruses.")

# Section 2~5: Data Loading & Preprocessing

In [ ]:
# ==================== SECTION 2: MULTI-VIRUS PROCESSING LOOP ====================
print("\n" + "="*70)
print("SECTION 2: STARTING MULTI-VIRUS PROCESSING")
print("="*70)

# Loop through each virus
for virus_idx, virus_name in enumerate(virus_list, 1):
    print(f"\n{'#'*70}")
    print(f"# PROCESSING VIRUS {virus_idx}/{len(virus_list)}: {virus_name}")
    print(f"{'#'*70}")
    
    try:
        # ==================== CREATE VIRUS-SPECIFIC OUTPUT FOLDER ====================
        virus_output_path = os.path.join(OUTPUT_FOLDER, virus_name)
        os.makedirs(virus_output_path, exist_ok=True)
        print(f"\n📁 Virus output folder: {virus_output_path}")
        
        # ==================== INITIALIZE VIRUS STATISTICS ====================
        virus_stats[virus_name] = {
            'total_spectra': 0,
            'concentrations': [],
            'concentration_pairs': [],
            'subgroups_created': 0,
            'combinations_generated': 0,
            'processing_time': None,
            'status': 'In Progress'
        }
        
        # Record start time for this virus
        virus_start_time = datetime.now()
        
        print(f"\n✅ Section 2 initialized for {virus_name}")
        print(f"   Ready to proceed to Section 3 (Data Loading)")
        
        # ==================== SECTION 3: DATA LOADING & PREPROCESSING ====================
        print(f"\n{'─'*70}")
        print(f"SECTION 3: DATA LOADING & PREPROCESSING FOR {virus_name}")
        print(f"{'─'*70}")
        
        # Load the CSV file
        print(f"\n📂 Loading data from: {INPUT_CSV}")
        df = pd.read_csv(INPUT_CSV)
        print(f"   Raw data shape: {df.shape}")
        
        # Extract wavenumber columns (400 to 1800)
        wavenumber_cols = [col for col in df.columns if col.isdigit() and 400 <= int(col) <= 1800]
        wavenumber_cols.sort(key=int)  # Sort numerically
        print(f"   Found {len(wavenumber_cols)} wavenumber columns: {wavenumber_cols[0]} to {wavenumber_cols[-1]}")
        
        # Extract and clean Label column (remove brackets and quotes)
        def clean_label(label_str):
            """Extract label from format like \"['SARSCoV2']\" to \"SARSCoV2\" """
            if pd.isna(label_str):
                return None
            # Remove brackets and quotes
            cleaned = str(label_str).strip("[]'\"")
            return cleaned
        
        df['Label_cleaned'] = df['Label'].apply(clean_label)
        
        # Extract and clean Concentration column (remove brackets)
        def clean_concentration(conc_str):
            """Extract concentration from format like \"[100.0]\" to 100.0"""
            if pd.isna(conc_str):
                return None
            # Remove brackets and convert to float
            cleaned = str(conc_str).strip("[]")
            try:
                return float(cleaned)
            except ValueError:
                return None
        
        df['Conc_cleaned'] = df['Conc'].apply(clean_concentration)
        
        # Filter for current virus only
        df_virus = df[df['Label_cleaned'] == virus_name].copy()
        print(f"   Data shape after filtering for {virus_name}: {df_virus.shape}")
        
        if len(df_virus) == 0:
            raise ValueError(f"No data found for virus '{virus_name}'. Please check virus name spelling.")
        
        # Remove rows with missing concentrations
        df_virus = df_virus.dropna(subset=['Conc_cleaned'])
        print(f"   Data shape after removing missing concentrations: {df_virus.shape}")
        
        # Sort by concentration for easier processing
        df_virus = df_virus.sort_values('Conc_cleaned').reset_index(drop=True)
        
        # Update virus statistics
        virus_stats[virus_name]['total_spectra'] = len(df_virus)
        virus_stats[virus_name]['concentrations'] = sorted(df_virus['Conc_cleaned'].unique().tolist())
        
        # Show concentration distribution
        conc_counts = df_virus['Conc_cleaned'].value_counts().sort_index()
        print(f"\n📊 Concentration distribution for {virus_name}:")
        for conc, count in conc_counts.items():
            print(f"   {conc}: {count} spectra")
        
        print(f"\n✅ Section 3 complete for {virus_name}")
        print(f"   Loaded {len(df_virus)} spectra across {len(conc_counts)} concentrations")

        # ==================== SECTION 4: CONCENTRATION PAIR SELECTION ====================
        print(f"\n{'─'*70}")
        print(f"SECTION 4: CONCENTRATION PAIR SELECTION FOR {virus_name}")
        print(f"{'─'*70}")
        
        # Get all unique concentrations, sorted
        unique_concentrations = sorted(df_virus['Conc_cleaned'].unique())
        print(f"\n📊 Found {len(unique_concentrations)} unique concentrations: {unique_concentrations}")
        
        # Check if we have enough concentrations for N-sequential pairs
        if len(unique_concentrations) < N:
            warning_msg = f"Insufficient concentrations for {virus_name}: found {len(unique_concentrations)}, need at least {N}"
            print(f"\n⚠️  WARNING: {warning_msg}")
            virus_processing_log['warnings'].append(f"{virus_name}: {warning_msg}")
            raise ValueError(warning_msg)
        
        # Create N-sequential dilution pairs
        sequential_pairs = []
        for i in range(len(unique_concentrations) - N + 1):
            pair = tuple(unique_concentrations[i:i+N])
            sequential_pairs.append(pair)
        
        print(f"\n🔗 Created {len(sequential_pairs)} sequential dilution pairs (N={N}):")
        for i, pair in enumerate(sequential_pairs, 1):
            print(f"   Pair {i}: {pair}")
        
        # Update virus statistics
        virus_stats[virus_name]['concentration_pairs'] = [list(pair) for pair in sequential_pairs]
        
        # Save the pairs to a file for reference
        pairs_file = os.path.join(virus_output_path, "sequential_pairs.txt")
        with open(pairs_file, 'w') as f:
            f.write(f"Sequential Dilution Pairs for {virus_name} (N={N})\n")
            f.write("="*70 + "\n\n")
            f.write(f"Dataset type: {dataset_type}\n")
            f.write(f"K (spectra per subgroup): {K}\n")
            f.write(f"M (subgroups per concentration): {M}\n")
            f.write(f"Expected combinations per pair: {M}×{M} = {M*M}\n\n")
            f.write(f"Concentration Pairs:\n")
            f.write("-"*70 + "\n")
            for i, pair in enumerate(sequential_pairs, 1):
                f.write(f"Pair {i}: {pair}\n")
                # Show spectra count for each concentration in pair
                for conc in pair:
                    count = len(df_virus[df_virus['Conc_cleaned'] == conc])
                    f.write(f"   {conc}: {count} spectra\n")
                f.write("\n")
            f.write(f"\nTotal pairs: {len(sequential_pairs)}\n")
            f.write(f"All concentrations: {unique_concentrations}\n")
        
        print(f"\n💾 Sequential pairs saved to: sequential_pairs.txt")
        
        # Collect all unique concentrations from all pairs (for subgroup creation)
        all_concentrations_in_pairs = set()
        for pair in sequential_pairs:
            all_concentrations_in_pairs.update(pair)
        all_concentrations_in_pairs = sorted(all_concentrations_in_pairs)
        
        print(f"\n📋 Summary:")
        print(f"   Total concentration pairs: {len(sequential_pairs)}")
        print(f"   Unique concentrations involved: {len(all_concentrations_in_pairs)}")
        print(f"   Concentrations: {all_concentrations_in_pairs}")
        
        print(f"\n✅ Section 4 complete for {virus_name}")
        print(f"   Ready to create subgroups for {len(all_concentrations_in_pairs)} concentrations")

        # ==================== SECTION 5: SUBGROUP CREATION WITH REPLACEMENT SAMPLING ====================
        print(f"\n{'─'*70}")
        print(f"SECTION 5: SUBGROUP CREATION WITH REPLACEMENT SAMPLING FOR {virus_name}")
        print(f"{'─'*70}")
        
        # Set random seed if specified
        if RANDOM_SEED is not None:
            np.random.seed(RANDOM_SEED)
            print(f"🎲 Random seed set to: {RANDOM_SEED}")
        
        print(f"\n🔄 Sampling strategy: {SAMPLING_STRATEGY}")
        print(f"   K (spectra per subgroup): {K}")
        print(f"   M (subgroups per concentration): {M}")
        print(f"   Replacement allowed: {ALLOW_REPLACEMENT}")
        
        # Track all created files
        created_csv_files = []
        created_index_files = []
        created_avg_files = []
        
        # Track coverage statistics for this virus
        coverage_stats = {}
        
        # Process each concentration
        print(f"\n📦 Processing {len(all_concentrations_in_pairs)} concentrations...")
        
        for conc in all_concentrations_in_pairs:
            print(f"\n{'─'*50}")
            print(f"Processing concentration: {conc}")
            print(f"{'─'*50}")
            
            # Get data for this concentration
            conc_data = df_virus[df_virus['Conc_cleaned'] == conc].copy()
            total_spectra = len(conc_data)
            conc_data = conc_data.reset_index(drop=True)  # Reset to concentration-specific indices
            
            print(f"   Total spectra available: {total_spectra}")
            
            # Handle edge case: K >= total_spectra
            if K >= total_spectra:
                actual_K = total_spectra
                actual_M = 1
                print(f"   ⚠️  K ({K}) >= total spectra ({total_spectra})")
                print(f"   → Adjusting to: K={actual_K}, M={actual_M} (single group with all spectra)")
            else:
                actual_K = K
                actual_M = M
            
            # Initialize coverage tracking for this concentration
            spectrum_coverage = {i: [] for i in range(total_spectra)}  # Track which subgroups each spectrum appears in
            
            # ==================== STRATEGY A: FORCE-INCLUDE FIRST ====================
            print(f"\n   🎯 Applying Strategy A: Force-Include First")
            print(f"   Creating {actual_M} subgroups with {actual_K} spectra each...")
            
            # Create all subgroups
            all_subgroup_indices = []
            
            for subgroup_i in range(actual_M):
                if subgroup_i == 0:
                    # First subgroup: ensure coverage by distributing all spectra
                    # Shuffle all spectrum indices
                    all_indices = list(range(total_spectra))
                    np.random.shuffle(all_indices)
                    
                    # Take first actual_K indices (or all if actual_K == total_spectra)
                    subgroup_indices = all_indices[:actual_K]
                    
                    print(f"   Subgroup {subgroup_i+1}: Force-include strategy (shuffled selection)")
                else:
                    # Check which spectra haven't been included yet
                    all_sampled_so_far = set()
                    for prev_subgroup in all_subgroup_indices:
                        all_sampled_so_far.update(prev_subgroup)
                    
                    missing_spectra = set(range(total_spectra)) - all_sampled_so_far
                    
                    if len(missing_spectra) > 0:
                        # Force-include missing spectra first
                        missing_list = list(missing_spectra)
                        num_forced = min(len(missing_list), actual_K)
                        forced_indices = missing_list[:num_forced]
                        
                        # Fill remaining slots with random sampling (with replacement)
                        remaining_slots = actual_K - num_forced
                        if remaining_slots > 0:
                            random_indices = np.random.choice(total_spectra, size=remaining_slots, replace=True).tolist()
                            subgroup_indices = forced_indices + random_indices
                        else:
                            subgroup_indices = forced_indices
                        
                        print(f"   Subgroup {subgroup_i+1}: {num_forced} forced + {remaining_slots} random")
                    else:
                        # All spectra already covered, pure random sampling with replacement
                        subgroup_indices = np.random.choice(total_spectra, size=actual_K, replace=True).tolist()
                        print(f"   Subgroup {subgroup_i+1}: Pure random sampling (all spectra already covered)")
                
                all_subgroup_indices.append(subgroup_indices)
                
                # Update coverage tracking
                for idx in subgroup_indices:
                    spectrum_coverage[idx].append(subgroup_i + 1)
            
            # ==================== VALIDATE COVERAGE ====================
            all_covered_spectra = set()
            for subgroup in all_subgroup_indices:
                all_covered_spectra.update(subgroup)
            
            missing_spectra = set(range(total_spectra)) - all_covered_spectra
            
            if len(missing_spectra) == 0:
                print(f"   ✅ Coverage check: All {total_spectra} spectra appear at least once")
            else:
                print(f"   ❌ WARNING: {len(missing_spectra)} spectra never sampled: {missing_spectra}")
                warning_msg = f"{virus_name} - Conc {conc}: {len(missing_spectra)} spectra missing"
                virus_processing_log['warnings'].append(warning_msg)
            
            # Calculate coverage statistics
            appearances_count = [len(spectrum_coverage[i]) for i in range(total_spectra)]
            coverage_stats[conc] = {
                'total_spectra': total_spectra,
                'min_appearances': min(appearances_count),
                'max_appearances': max(appearances_count),
                'mean_appearances': np.mean(appearances_count),
                'expected_mean': (actual_M * actual_K) / total_spectra
            }
            
            print(f"\n   📊 Coverage statistics:")
            print(f"      Min appearances: {coverage_stats[conc]['min_appearances']}")
            print(f"      Max appearances: {coverage_stats[conc]['max_appearances']}")
            print(f"      Mean appearances: {coverage_stats[conc]['mean_appearances']:.2f}")
            print(f"      Expected mean: {coverage_stats[conc]['expected_mean']:.2f}")
            
            # ==================== SAVE SUBGROUPS ====================
            print(f"\n   💾 Saving subgroups...")
            
            for subgroup_i, subgroup_conc_indices in enumerate(all_subgroup_indices):
                # Get actual spectrum data
                subgroup_data = conc_data.iloc[subgroup_conc_indices].copy()
                original_indices = conc_data.iloc[subgroup_conc_indices].index.tolist()
                
                # ===== 1. Save subgroup CSV =====
                csv_data = pd.DataFrame()
                csv_data['wavenumber'] = [int(wn) for wn in wavenumber_cols]
                
                for spec_idx, (_, spectrum_row) in enumerate(subgroup_data.iterrows()):
                    col_name = f"spectrum_{spec_idx}"
                    csv_data[col_name] = spectrum_row[wavenumber_cols].values
                
                csv_filename = f"{virus_name}_{conc}_subgroup{subgroup_i+1}.csv"
                csv_filepath = os.path.join(virus_output_path, csv_filename)
                csv_data.to_csv(csv_filepath, index=False)
                created_csv_files.append(csv_filepath)
                
                # ===== 2. Save indices file with coverage info =====
                indices_filename = f"{virus_name}_{conc}_subgroup{subgroup_i+1}_indices.txt"
                indices_filepath = os.path.join(virus_output_path, indices_filename)
                
                with open(indices_filepath, 'w') as f:
                    f.write(f"Subgroup indices for {virus_name} concentration {conc}, subgroup {subgroup_i+1}\n")
                    f.write("="*70 + "\n\n")
                    f.write(f"Sampling method: {SAMPLING_STRATEGY}\n")
                    f.write(f"Random seed: {RANDOM_SEED}\n")
                    f.write(f"Replacement allowed: {ALLOW_REPLACEMENT}\n")
                    f.write(f"Concentration: {conc}\n")
                    f.write(f"Number of spectra in subgroup: {len(subgroup_conc_indices)}\n")
                    f.write(f"Subgroup: {subgroup_i+1} of {actual_M}\n\n")
                    
                    f.write("Index mapping:\n")
                    f.write("Concentration-specific indices: " + str(subgroup_conc_indices) + "\n")
                    f.write("Original DataFrame indices: " + str(original_indices) + "\n\n")
                    
                    f.write("Individual spectrum mapping:\n")
                    for i, conc_idx in enumerate(subgroup_conc_indices):
                        appearances = len(spectrum_coverage[conc_idx])
                        f.write(f"  spectrum_{i}: conc_index_{conc_idx} (appears in {appearances} subgroups total)\n")
                
                created_index_files.append(indices_filepath)
                
                # ===== 3. Calculate and save average spectrum =====
                spectral_data = csv_data[[col for col in csv_data.columns if col.startswith('spectrum_')]].values
                average_spectrum = np.mean(spectral_data, axis=1)
                
                avg_data = pd.DataFrame()
                avg_data['wavenumber'] = csv_data['wavenumber']
                avg_data['average_spectrum'] = average_spectrum
                
                avg_filename = f"{virus_name}_{conc}_subgroup{subgroup_i+1}_avg.csv"
                avg_filepath = os.path.join(virus_output_path, avg_filename)
                avg_data.to_csv(avg_filepath, index=False)
                created_avg_files.append(avg_filepath)
            
            print(f"   ✅ Created {actual_M} subgroups for concentration {conc}")
            print(f"      Files: {actual_M} CSV + {actual_M} indices + {actual_M} averages")
        
        # ==================== UPDATE VIRUS STATISTICS ====================
        virus_stats[virus_name]['subgroups_created'] = len(created_csv_files)
        virus_stats[virus_name]['combinations_generated'] = len(sequential_pairs) * (M * M)
        virus_stats[virus_name]['coverage_stats'] = coverage_stats
        
        # Calculate processing time
        virus_end_time = datetime.now()
        processing_time = (virus_end_time - virus_start_time).total_seconds()
        virus_stats[virus_name]['processing_time'] = f"{processing_time:.2f}s"
        virus_stats[virus_name]['status'] = 'Success'
        
        # Mark as successful
        virus_processing_log['successful'].append(virus_name)
        
        print(f"\n{'='*70}")
        print(f"✅ COMPLETED PROCESSING FOR {virus_name}")
        print(f"{'='*70}")
        print(f"   Total subgroups created: {len(created_csv_files)}")
        print(f"   Total combinations: {virus_stats[virus_name]['combinations_generated']}")
        print(f"   Processing time: {processing_time:.2f}s")
        print(f"   Output location: {virus_output_path}")
        
    except Exception as e:
        # Error handling for this virus
        error_msg = f"Failed to initialize processing for {virus_name}: {str(e)}"
        print(f"\n❌ ERROR: {error_msg}")
        virus_processing_log['failed'].append(virus_name)
        virus_stats[virus_name] = {
            'status': 'Failed',
            'error': str(e)
        }
        continue  # Skip to next virus

print(f"\n{'='*70}")
print("SECTION 2~5: MULTI-VIRUS LOOP STRUCTURE READY")

# SECTION 6: MULTI-VIRUS SUMMARY REPORT

In [ ]:
# ==================== SECTION 6: VALIDATION & MULTI-VIRUS SUMMARY REPORT ====================
print("\n" + "="*70)
print("SECTION 6: VALIDATION & SUMMARY REPORT")
print("="*70)

# ==================== PART A: DETAILED VALIDATION ====================
print("\n" + "─"*70)
print("PART A: VALIDATION & QUALITY CHECKS")
print("─"*70)

validation_results = []

for virus_name in virus_processing_log['successful']:
    print(f"\n🔍 Validating {virus_name}...")
    
    virus_output_path = os.path.join(OUTPUT_FOLDER, virus_name)
    
    # Check if coverage_stats exists for this virus
    if 'coverage_stats' not in virus_stats[virus_name]:
        print(f"   ⚠️  No coverage stats found for {virus_name}")
        continue
    
    coverage_stats = virus_stats[virus_name]['coverage_stats']
    
    for conc, stats in coverage_stats.items():
        print(f"\n   Concentration: {conc}")
        
        # === 1. Coverage Validation ===
        total_spectra = stats['total_spectra']
        
        # Re-check coverage by reading the indices files
        subgroup_files = [f for f in os.listdir(virus_output_path) 
                         if f.startswith(f"{virus_name}_{conc}_subgroup") and f.endswith("_indices.txt")]
        
        # Collect all unique spectra indices that were used
        all_used_indices = set()
        for indices_file in subgroup_files:
            filepath = os.path.join(virus_output_path, indices_file)
            with open(filepath, 'r') as f:
                content = f.read()
                # Extract concentration-specific indices
                import re
                match = re.search(r'Concentration-specific indices: \[(.*?)\]', content)
                if match:
                    indices_str = match.group(1)
                    if indices_str.strip():
                        indices = [int(x.strip()) for x in indices_str.split(',')]
                        all_used_indices.update(indices)
        
        spectra_selected_once = len(all_used_indices)
        spectra_never_selected = total_spectra - spectra_selected_once
        coverage_percentage = (spectra_selected_once / total_spectra * 100) if total_spectra > 0 else 0
        
        # === 2. Subgroup Validation ===
        csv_files = [f for f in os.listdir(virus_output_path) 
                    if f.startswith(f"{virus_name}_{conc}_subgroup") and f.endswith(".csv") and not f.endswith("_avg.csv")]
        avg_files = [f for f in os.listdir(virus_output_path) 
                    if f.startswith(f"{virus_name}_{conc}_subgroup") and f.endswith("_avg.csv")]
        
        expected_subgroups = M if K < total_spectra else 1
        actual_subgroups = len(csv_files)
        
        # === 3. File Integrity Check ===
        files_match = (len(csv_files) == len(subgroup_files) == len(avg_files))
        
        # === 4. Subgroup Size Validation ===
        subgroup_sizes_correct = True
        for csv_file in csv_files:
            csv_path = os.path.join(virus_output_path, csv_file)
            df_check = pd.read_csv(csv_path)
            spectrum_cols = [col for col in df_check.columns if col.startswith('spectrum_')]
            expected_size = K if K < total_spectra else total_spectra
            if len(spectrum_cols) != expected_size:
                subgroup_sizes_correct = False
                print(f"      ❌ {csv_file}: has {len(spectrum_cols)} spectra, expected {expected_size}")
        
        # === 5. Determine Status ===
        if coverage_percentage == 100 and files_match and subgroup_sizes_correct and actual_subgroups == expected_subgroups:
            status = "✅ PASS"
        else:
            status = "⚠️ ISSUES"
        
        # === 6. Print Validation Results ===
        print(f"      Total spectra: {total_spectra}")
        print(f"      Spectra selected ≥1 time: {spectra_selected_once}")
        print(f"      Spectra never selected: {spectra_never_selected}")
        print(f"      Coverage: {coverage_percentage:.1f}%")
        print(f"      Min appearances: {stats['min_appearances']}")
        print(f"      Max appearances: {stats['max_appearances']}")
        print(f"      Mean appearances: {stats['mean_appearances']:.2f}")
        print(f"      Expected subgroups: {expected_subgroups}")
        print(f"      Actual subgroups: {actual_subgroups}")
        print(f"      Files match: {files_match}")
        print(f"      Subgroup sizes correct: {subgroup_sizes_correct}")
        print(f"      Status: {status}")
        
        # === 7. Store in validation results ===
        validation_results.append({
            'Virus': virus_name,
            'Concentration': conc,
            'Total_Spectra': total_spectra,
            'Spectra_Selected_Once': spectra_selected_once,
            'Spectra_Never_Selected': spectra_never_selected,
            'Coverage_Percentage': round(coverage_percentage, 2),
            'Min_Appearances': stats['min_appearances'],
            'Max_Appearances': stats['max_appearances'],
            'Mean_Appearances': round(stats['mean_appearances'], 2),
            'Expected_Mean_Appearances': round(stats['expected_mean'], 2),
            'Expected_Subgroups': expected_subgroups,
            'Actual_Subgroups': actual_subgroups,
            'Files_Match': files_match,
            'Subgroup_Sizes_Correct': subgroup_sizes_correct,
            'Status': status
        })

# ==================== PART B: SUMMARY REPORTS ====================
print("\n" + "─"*70)
print("PART B: GENERATING SUMMARY REPORTS")
print("─"*70)

# === 1. Create Validation Table (CSV) ===
validation_df = pd.DataFrame(validation_results)
validation_csv_path = os.path.join(OUTPUT_FOLDER, "validation_table.csv")
validation_df.to_csv(validation_csv_path, index=False)
print(f"\n💾 Validation table saved: validation_table.csv")

# === 2. Create Multi-Virus Summary Report (TXT) ===
summary_file = os.path.join(OUTPUT_FOLDER, "multi_virus_summary.txt")

with open(summary_file, 'w') as f:
    # Header
    f.write("="*70 + "\n")
    f.write("MULTI-VIRUS SUBGROUP BUILDER - SUMMARY REPORT\n")
    f.write("="*70 + "\n\n")
    
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Dataset type: {dataset_type}\n")
    f.write(f"Configuration: K={K}, M={M}, N={N}\n")
    f.write(f"Sampling strategy: {SAMPLING_STRATEGY}\n")
    f.write(f"Random seed: {RANDOM_SEED}\n\n")
    
    # Overall Statistics
    f.write("="*70 + "\n")
    f.write("OVERALL STATISTICS\n")
    f.write("="*70 + "\n\n")
    f.write(f"Total viruses requested: {len(virus_list)}\n")
    f.write(f"Successfully processed: {len(virus_processing_log['successful'])}\n")
    f.write(f"Failed: {len(virus_processing_log['failed'])}\n")
    f.write(f"Warnings: {len(virus_processing_log['warnings'])}\n\n")
    
    # Successful Viruses
    if virus_processing_log['successful']:
        f.write("Successful viruses:\n")
        for virus in virus_processing_log['successful']:
            f.write(f"  ✅ {virus}\n")
        f.write("\n")
    
    # Failed Viruses
    if virus_processing_log['failed']:
        f.write("Failed viruses:\n")
        for virus in virus_processing_log['failed']:
            f.write(f"  ❌ {virus}\n")
            if virus in virus_stats and 'error' in virus_stats[virus]:
                f.write(f"     Error: {virus_stats[virus]['error']}\n")
        f.write("\n")
    
    # Warnings
    if virus_processing_log['warnings']:
        f.write("Warnings:\n")
        for warning in virus_processing_log['warnings']:
            f.write(f"  ⚠️  {warning}\n")
        f.write("\n")
    
    # Per-Virus Details
    f.write("="*70 + "\n")
    f.write("PER-VIRUS DETAILS\n")
    f.write("="*70 + "\n\n")
    
    for virus_name in virus_processing_log['successful']:
        stats = virus_stats[virus_name]
        f.write(f"\n{virus_name}\n")
        f.write("-"*70 + "\n")
        f.write(f"Status: {stats['status']}\n")
        f.write(f"Processing time: {stats['processing_time']}\n")
        f.write(f"Total spectra: {stats['total_spectra']}\n")
        f.write(f"Concentrations: {stats['concentrations']}\n")
        f.write(f"Concentration pairs: {len(stats['concentration_pairs'])}\n")
        f.write(f"Subgroups created: {stats['subgroups_created']}\n")
        f.write(f"Total combinations: {stats['combinations_generated']}\n")
        
        # Coverage details per concentration
        if 'coverage_stats' in stats:
            f.write(f"\nCoverage by concentration:\n")
            for conc, cov_stats in stats['coverage_stats'].items():
                f.write(f"  {conc}:\n")
                f.write(f"    Total spectra: {cov_stats['total_spectra']}\n")
                f.write(f"    Min appearances: {cov_stats['min_appearances']}\n")
                f.write(f"    Max appearances: {cov_stats['max_appearances']}\n")
                f.write(f"    Mean appearances: {cov_stats['mean_appearances']:.2f}\n")
        f.write("\n")
    
    # Validation Summary
    f.write("="*70 + "\n")
    f.write("VALIDATION SUMMARY\n")
    f.write("="*70 + "\n\n")
    
    total_concentrations = len(validation_results)
    passed = sum(1 for r in validation_results if r['Status'] == "✅ PASS")
    issues = total_concentrations - passed
    
    f.write(f"Total (virus, concentration) combinations validated: {total_concentrations}\n")
    f.write(f"Passed all checks: {passed}\n")
    f.write(f"Have issues: {issues}\n\n")
    
    if issues > 0:
        f.write("Combinations with issues:\n")
        for r in validation_results:
            if r['Status'] != "✅ PASS":
                f.write(f"  {r['Virus']} - {r['Concentration']}: ")
                issues_list = []
                if r['Coverage_Percentage'] < 100:
                    issues_list.append(f"Coverage {r['Coverage_Percentage']:.1f}%")
                if r['Actual_Subgroups'] != r['Expected_Subgroups']:
                    issues_list.append(f"Subgroups {r['Actual_Subgroups']}/{r['Expected_Subgroups']}")
                if not r['Files_Match']:
                    issues_list.append("File mismatch")
                if not r['Subgroup_Sizes_Correct']:
                    issues_list.append("Size errors")
                f.write(", ".join(issues_list) + "\n")
        f.write("\n")
    
    # Output Location
    f.write("="*70 + "\n")
    f.write("OUTPUT LOCATION\n")
    f.write("="*70 + "\n\n")
    f.write(f"Main output folder: {OUTPUT_FOLDER}\n")
    f.write(f"Validation table: validation_table.csv\n")
    f.write(f"Summary report: multi_virus_summary.txt\n\n")
    
    for virus_name in virus_processing_log['successful']:
        f.write(f"{virus_name}: {os.path.join(OUTPUT_FOLDER, virus_name)}\n")

print(f"💾 Summary report saved: multi_virus_summary.txt")

# === 3. Console Summary ===
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
print(f"\n📊 Processing Complete:")
print(f"   Total viruses: {len(virus_list)}")
print(f"   ✅ Successful: {len(virus_processing_log['successful'])}")
print(f"   ❌ Failed: {len(virus_processing_log['failed'])}")
print(f"   ⚠️  Warnings: {len(virus_processing_log['warnings'])}")

print(f"\n📋 Validation Results:")
total_concentrations = len(validation_results)
passed = sum(1 for r in validation_results if r['Status'] == "✅ PASS")
print(f"   Total (virus, conc) combinations: {total_concentrations}")
print(f"   ✅ Passed all checks: {passed}/{total_concentrations}")
if passed < total_concentrations:
    print(f"   ⚠️  Have issues: {total_concentrations - passed}/{total_concentrations}")

print(f"\n📁 Output Files:")
print(f"   Main folder: {OUTPUT_FOLDER}")
print(f"   Validation table: validation_table.csv")
print(f"   Summary report: multi_virus_summary.txt")

if virus_processing_log['successful']:
    print(f"\n🦠 Successfully processed viruses:")
    for virus in virus_processing_log['successful']:
        print(f"   ✅ {virus}")

if virus_processing_log['failed']:
    print(f"\n❌ Failed viruses:")
    for virus in virus_processing_log['failed']:
        print(f"   ❌ {virus}")

print(f"\n{'='*70}")
print("✅ ALL DONE! Multi-virus subgroup creation complete.")
print(f"{'='*70}")
print(f"\nPlease review:")
print(f"1. validation_table.csv - for detailed per-concentration validation")
print(f"2. multi_virus_summary.txt - for full summary report")
print(f"3. Individual virus folders - for subgroup files")

In [ ]:
# Section 5: Average Calculation & Saving

print(f"Calculating averages for {len(created_csv_files)} subgroups...")

# Track created average files
created_avg_files = []

# Process each CSV file to create its average
for csv_file in created_csv_files:
    print(f"\nProcessing: {os.path.basename(csv_file)}")
    
    # Load the subgroup CSV
    subgroup_data = pd.read_csv(csv_file)
    
    # Extract wavenumbers and spectral data
    wavenumbers = subgroup_data['wavenumber'].values
    spectrum_cols = [col for col in subgroup_data.columns if col.startswith('spectrum_')]
    n_spectra = len(spectrum_cols)
    
    print(f"  Found {n_spectra} spectra with {len(wavenumbers)} wavenumbers")
    
    # Calculate average spectrum
    spectral_data = subgroup_data[spectrum_cols].values  # Shape: (wavenumbers, spectra)
    average_spectrum = np.mean(spectral_data, axis=1)  # Average across spectra (axis=1)
    
    print(f"  Calculated average spectrum")
    
    # Create average CSV with same format as subgroup CSV
    avg_data = pd.DataFrame()
    avg_data['wavenumber'] = wavenumbers
    avg_data['average_spectrum'] = average_spectrum
    
    # Generate average filename
    avg_filename = csv_file.replace('.csv', '_avg.csv')
    avg_data.to_csv(avg_filename, index=False)
    created_avg_files.append(avg_filename)
    
    print(f"  Saved average to: {os.path.basename(avg_filename)}")

print(f"\n✅ Average calculation complete!")
print(f"Created {len(created_avg_files)} average files")

# ==================== TESTING & VALIDATION ====================
if TEST_MODE:
    print("\n=== AVERAGE CALCULATION VALIDATION ===")
    
    # Check file count
    print(f"📊 File count validation:")
    print(f"   Original subgroup files: {len(created_csv_files)}")
    print(f"   Average files created: {len(created_avg_files)}")
    
    if len(created_avg_files) == len(created_csv_files):
        print(f"   ✅ Average file count matches subgroup file count")
    else:
        print(f"   ❌ Mismatch in file counts")
    
    # Validate average files contain correct wavenumbers
    print(f"\n🌊 Wavenumber validation:")
    expected_wavenumbers = [int(wn) for wn in wavenumber_cols]
    
    wavenumber_issues = []
    for avg_file in created_avg_files[:3]:  # Check first 3 files
        avg_data = pd.read_csv(avg_file)
        actual_wavenumbers = avg_data['wavenumber'].tolist()
        
        if actual_wavenumbers == expected_wavenumbers:
            print(f"   ✅ {os.path.basename(avg_file)}: wavenumbers correct")
        else:
            print(f"   ❌ {os.path.basename(avg_file)}: wavenumber mismatch")
            wavenumber_issues.append(avg_file)
    
    if len(created_avg_files) > 3:
        print(f"   ... (checked first 3 files, {len(created_avg_files)-3} more files created)")
    
    # Manual validation of average calculation (detailed check for first file)
    print(f"\n🧮 Manual average calculation validation:")
    if created_csv_files:
        # Use first subgroup file for detailed validation
        test_csv = created_csv_files[0]
        test_avg = test_csv.replace('.csv', '_avg.csv')
        
        print(f"   Validating: {os.path.basename(test_csv)}")
        
        # Load original subgroup data
        original_data = pd.read_csv(test_csv)
        spectrum_cols = [col for col in original_data.columns if col.startswith('spectrum_')]
        
        # Load calculated average
        avg_data = pd.read_csv(test_avg)
        calculated_avg = avg_data['average_spectrum'].values
        
        # Manually calculate average
        manual_avg = np.mean(original_data[spectrum_cols].values, axis=1)
        
        # Compare
        max_diff = np.max(np.abs(calculated_avg - manual_avg))
        mean_diff = np.mean(np.abs(calculated_avg - manual_avg))
        
        print(f"   Number of spectra averaged: {len(spectrum_cols)}")
        print(f"   Maximum difference: {max_diff:.10f}")
        print(f"   Mean absolute difference: {mean_diff:.10f}")
        
        if max_diff < 1e-10:  # Very small tolerance for floating point precision
            print(f"   ✅ Average calculation is correct (within floating point precision)")
        else:
            print(f"   ❌ Average calculation has significant differences")
            
            # Show sample comparison for debugging
            print(f"   Sample comparison (first 5 wavenumbers):")
            for i in range(min(5, len(calculated_avg))):
                wn = original_data['wavenumber'].iloc[i]
                calc = calculated_avg[i]
                manual = manual_avg[i]
                diff = abs(calc - manual)
                print(f"     {wn} cm⁻¹: calculated={calc:.6f}, manual={manual:.6f}, diff={diff:.10f}")
    
    # Cross-reference with original subgroup files
    print(f"\n🔗 Cross-reference validation:")
    sample_checks = min(3, len(created_avg_files))
    
    for i in range(sample_checks):
        csv_file = created_csv_files[i]
        avg_file = created_avg_files[i]
        
        # Extract concentration and subgroup from filename
        basename = os.path.basename(csv_file)
        # Expected format: SARSCoV2_{conc}_subgroup{i}.csv
        
        original_data = pd.read_csv(csv_file)
        avg_data = pd.read_csv(avg_file)
        
        spectrum_cols = [col for col in original_data.columns if col.startswith('spectrum_')]
        
        print(f"   {basename}:")
        print(f"     Original spectra: {len(spectrum_cols)}")
        print(f"     Average file exists: {os.path.exists(avg_file)}")
        print(f"     Average data points: {len(avg_data)}")
        print(f"     ✅ Files correspond correctly")
    
    # Summary statistics for all averages
    print(f"\n📈 Average spectrum statistics (sample from first file):")
    if created_avg_files:
        first_avg = pd.read_csv(created_avg_files[0])
        avg_spectrum = first_avg['average_spectrum'].values
        
        print(f"   Min intensity: {np.min(avg_spectrum):.6f}")
        print(f"   Max intensity: {np.max(avg_spectrum):.6f}")
        print(f"   Mean intensity: {np.mean(avg_spectrum):.6f}")
        print(f"   Std intensity: {np.std(avg_spectrum):.6f}")
        
        # Check for any obvious issues
        if np.any(np.isnan(avg_spectrum)):
            print(f"   ❌ WARNING: NaN values found in average spectrum")
        elif np.any(np.isinf(avg_spectrum)):
            print(f"   ❌ WARNING: Infinite values found in average spectrum")
        else:
            print(f"   ✅ No NaN or infinite values detected")

print(f"\n✅ Section 5 complete. All averages calculated and validated.")
print(f"Processing summary:")
print(f"  - Processed {len(all_concentrations_in_pairs)} concentrations")
print(f"  - Created {len(created_csv_files)} subgroups")
print(f"  - Calculated {len(created_avg_files)} averages")
print(f"  - All files saved to: {OUTPUT_PATH}")